# T01: Hello, Spindle!

## What You'll Learn

Spindle is a synthetic data generation library that produces realistic, relationally-consistent datasets across multiple business domains. In this notebook you will:

1. **Install** the `sqllocks-spindle` package
2. **Generate** a complete retail dataset with a single function call
3. **Verify** that all foreign-key relationships are intact
4. **Inspect** the generated DataFrames
5. **Export** everything to CSV files

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle` (we'll do this in the first code cell)

## Time Estimate

**~5 minutes** from start to finish.

In [1]:
# Uncomment the line below if you're running in Microsoft Fabric
# or any environment where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Import Spindle and Create a Retail Dataset

### What we're about to do
We'll import two things:
- **`Spindle`** — the main entry point that orchestrates data generation.
- **`RetailDomain`** — a pre-built schema definition containing tables like `customer`, `product`, `order`, and `order_line`, all wired together with proper foreign keys.

We'll then call `Spindle.generate()` with the `fabric_demo` scale (a small, fast dataset perfect for tutorials) and a fixed `seed` so your results match ours exactly.

### Why this matters
A single call to `generate()` produces an entire relational database's worth of data — no boilerplate, no manual wiring. The seed ensures reproducibility, which is critical for testing and demos.

### What to expect
You'll see a summary showing every table that was generated, along with its row count.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

# Generate a retail dataset at the "fabric_demo" scale
spindle = Spindle()
result = spindle.generate(
    domain=RetailDomain(),
    scale="fabric_demo",
    seed=42,
)

print(result.summary())

Spindle Generation Result
Schema: retail_3nf
Domain: retail
Mode:   3nf
Seed:   42
Time:   0.2s

Table                             Rows  Columns
---------------------------------------------
customer                           200        8
address                            300       10
product_category                    50        4
product                            100        6
promotion                          200        6
store                              150        5
order                            1,000        8
order_line                       2,500        8
return                             170        5
---------------------------------------------
TOTAL                            4,670


## Verify Foreign-Key Integrity

### What we're about to do
We'll call `result.verify_integrity()` which checks every foreign-key relationship in the generated dataset. It returns a list of violations — we expect that list to be empty.

### Why this matters
Synthetic data is only useful if it's relationally valid. If an `order` references a `customer_id` that doesn't exist in the `customer` table, your downstream queries, dashboards, and ML pipelines will break in confusing ways. Spindle guarantees referential integrity by design, and this call lets you prove it.

### What to expect
An empty list of violations and a confirmation message.

In [3]:
violations = result.verify_integrity()

assert len(violations) == 0, f"Found {len(violations)} FK violations!"

print(f"Checked all foreign-key relationships.")
print(f"Violations found: {len(violations)}")
print("All FK relationships verified!")

Checked all foreign-key relationships.
Violations found: 0
All FK relationships verified!


## Inspect a Table

### What we're about to do
We'll grab the `customer` DataFrame from the result and look at:
- **`head()`** — the first few rows of data
- **`shape`** — the number of rows and columns
- **`dtypes`** — the data types of each column

### Why this matters
Every table Spindle generates is a standard pandas DataFrame. That means you can use the full pandas API for filtering, grouping, joining, and plotting — no special adapters needed.

### What to expect
You'll see a preview of the customer data with realistic names, emails, and other attributes, followed by the shape and column types.

In [4]:
customers = result.tables["customer"]

print("=== Customers — First 5 Rows ===")
print(customers.head())
print(f"\nShape: {customers.shape[0]} rows x {customers.shape[1]} columns")
print(f"\n=== Column Data Types ===")
print(customers.dtypes)

=== Customers — First 5 Rows ===
   customer_id first_name  last_name                  email gender  \
0            1     Dillon  Macdonald     hyoung@example.org      F   
1            2     Andrew       Paul  brianna69@example.com      M   
2            3    Jasmine   Schaefer     lsmith@example.net      M   
3            4       John      Evans   walvarez@example.net      M   
4            5     Andrea      Ramos   teresa60@example.org      M   

  loyalty_tier                   signup_date is_active  
0       Silver 2025-08-11 20:36:31.081033514      true  
1        Basic 2024-11-18 05:11:09.502839751      true  
2     Platinum 2025-11-07 06:55:16.583412953      true  
3         Gold 2025-02-15 17:27:40.257596628      true  
4        Basic 2025-06-19 12:43:20.628112464      true  

Shape: 200 rows x 8 columns

=== Column Data Types ===
customer_id              int64
first_name                 str
last_name                  str
email                      str
gender                  

## Export to CSV

### What we're about to do
We'll call `result.to_csv()` to write every generated table to its own CSV file inside a `./spindle_output` directory.

### Why this matters
CSV export is the fastest way to get your synthetic data into other tools — Power BI, Excel, a Lakehouse, or another notebook. One call writes all tables, correctly named.

### What to expect
A list of the CSV files that were created, one per table.

In [5]:
import os

output_dir = "./spindle_output"
result.to_csv(output_dir)

files = sorted(os.listdir(output_dir))
print(f"Exported {len(files)} CSV files to {output_dir}/\n")
for f in files:
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f} ({size:,} bytes)")

Exported 9 CSV files to ./spindle_output/

  address.csv (24,237 bytes)
  customer.csv (16,874 bytes)
  order.csv (60,710 bytes)
  order_line.csv (83,244 bytes)
  product.csv (4,692 bytes)
  product_category.csv (1,225 bytes)
  promotion.csv (19,590 bytes)
  return.csv (9,766 bytes)
  store.csv (6,242 bytes)


## What's Next?

You've just generated, verified, inspected, and exported a complete retail dataset in under 5 minutes. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T02: Explore All Domains** | See all 12 built-in domains, compare their schemas and table counts |
| **T03: Custom Schema** | Define your own tables, columns, and relationships from scratch |
| **CLI Cheatsheet** | Generate data from the command line without writing any Python |

Happy generating!